# H18b: Is the ~30x Muon Constant Factor LR-Independent?\n\n## Motivation and Scientific Context\n\nExperiment H18a established that, after properly controlling for learning rate, the Muon optimizer\nretains a **constant ~30x advantage** over SGD across network depths. This was a critical finding\nfrom the D-test retraction: the advantage is *not* exponential in depth (which would have suggested\na gauge-fixing mechanism compounding layer-by-layer), but rather a stable multiplicative factor.\n\nHowever, a crucial methodological concern remains: **was that 30x measured at sufficient LR resolution?**\n\nIf we only tested a coarse grid of learning rates (e.g., 10 candidates), the \"optimal\" LR for each\noptimizer might not be truly optimal. A finer sweep could reveal that SGD's best LR was missed,\nor that Muon's advantage shrinks/grows with more careful tuning. The ~30x could be an artifact of\nLR grid granularity rather than a genuine intrinsic property of the Newton-Schulz orthogonalization.\n\n## Hypothesis\n\n> **The ~30x constant advantage is stable across LR granularities.** With exhaustive LR sweeps\n> (50 log-spaced candidates per optimizer), the advantage ratio SGD_best / Muon_best remains\n> in the range [15x, 60x] for all depths L in {2, 4, 8, 16}, confirming it is a genuine\n> constant-factor directional benefit of the Newton-Schulz steepest descent direction.\n\n## Protocol\n\n| Parameter | Value |\n|-----------|-------|\n| Depths tested | L = 2, 4, 8, 16 |\n| SGD LR sweep | 50 candidates, log-spaced in [1e-6, 0.5] |\n| Muon LR sweep | 50 candidates, log-spaced in [1e-5, 0.2] |\n| Selection phase | 3 seeds per LR, pick best by median loss |\n| Evaluation phase | 5 seeds at best LR, report mean loss |\n| Training steps | 300 per run |\n| Matrix dimension | 32x32 |\n\n## Pass Criteria\n\n1. **T1 (Bounded range):** All advantage ratios fall in [5x, 100x] -- same order of magnitude\n2. **T2 (Low variation):** Coefficient of variation of log(advantage) across depths < 0.3\n3. **T3 (No depth trend):** Spearman correlation between depth and advantage is not significant (|rho| < 0.8)\n\n## Why This Matters\n\nIf the ~30x factor is genuinely LR-independent, it tells us something fundamental: the Newton-Schulz\northogonalization provides a *fixed geometric improvement* in the update direction, regardless of\nhow carefully you tune step size. This is consistent with the interpretation that Muon projects\ngradients onto a better-conditioned subspace, yielding a constant-factor speedup that cannot be\nrecovered by SGD through LR tuning alone.

In [ ]:
"""
H18b: Is the ~30x Constant Factor LR-Independent?
===================================================

FROM D-TEST RETRACTION: After controlling for LR, the Muon advantage is a
constant ~30x across depths (NOT exponential). But was that 30x measured at
a SINGLE LR grid resolution? The number could shift with exhaustive sweeps.

HYPOTHESIS:
  The ~30x constant advantage is stable across LR granularities. If you do
  EXHAUSTIVE LR sweeps (50+ candidates, fine-grained) at each depth, the
  advantage ratio SGD/Muon remains in the range [15x, 60x] for all depths
  2-16, confirming it is a genuine constant-factor directional benefit.

PROTOCOL:
  For depths L in {2, 4, 8, 16}:
    - SGD: sweep 50 LRs log-spaced in [1e-6, 0.5]
    - Muon: sweep 50 LRs log-spaced in [1e-5, 0.2]
    - 5 seeds each, pick best by median
    - Compute advantage ratio at best LRs
  Measure coefficient of variation of advantage across depths.

PASS CRITERIA:
  - All advantages in [5x, 100x] (same order of magnitude)
  - CV of log(advantage) across depths < 0.3
  - No monotone trend: Spearman(depth, advantage) not significant
"""

## Section 1: Imports and Environment Setup

In [ ]:
import numpy as np
import os

print("H18b: 30x Factor LR-Independence Test")
print(f"NumPy version: {np.__version__}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

## Section 2: Experimental Configuration\n\nThe key design decision here is the **LR grid density**. Previous experiments (H18a) used\nrelatively coarse grids (~10-20 candidates). Here we use **50 candidates per optimizer**,\nlog-spaced to cover the full viable range.\n\nWhy log-spacing? Learning rate sensitivity is multiplicative, not additive. The difference\nbetween LR=0.001 and LR=0.002 matters as much as the difference between LR=0.01 and LR=0.02.\nLog-spacing ensures uniform resolution in this multiplicative sense.\n\nThe SGD and Muon grids differ in range because:\n- **SGD** can tolerate larger LRs (up to 0.5) since raw gradients have bounded norm for our problem\n- **Muon** uses Newton-Schulz-orthogonalized updates (unit spectral norm), so its effective step\n  scale differs and the viable range is shifted

In [ ]:
DIM = 32
DEPTHS = [2, 4, 8, 16]
NUM_STEPS = 300
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 5
BATCH_SIZE = 64

print("=== Experimental Configuration ===")
print(f"  Matrix dimension:        {DIM}x{DIM}")
print(f"  Depths to test:          {DEPTHS}")
print(f"  Training steps per run:  {NUM_STEPS}")
print(f"  Momentum coefficient:    {MOMENTUM}")
print(f"  Newton-Schulz iterations:{NS_ITERS}")
print(f"  Seeds for evaluation:    {NUM_SEEDS}")
print(f"  Batch size:              {BATCH_SIZE}")
print(f"")
print(f"  Total optimizer configs: {len(DEPTHS)} depths x 2 optimizers = {len(DEPTHS)*2}")
print(f"  Total LR candidates:     50 (SGD) + 50 (Muon) = 100 per depth")
print(f"  Selection-phase runs:    100 LRs x 3 seeds x {len(DEPTHS)} depths = {100*3*len(DEPTHS)}")
print(f"  Eval-phase runs:         2 optimizers x {NUM_SEEDS} seeds x {len(DEPTHS)} depths = {2*NUM_SEEDS*len(DEPTHS)}")
print(f"  Grand total runs:        ~{100*3*len(DEPTHS) + 2*NUM_SEEDS*len(DEPTHS)}")

In [ ]:
# EXHAUSTIVE grids: 50 candidates each
SGD_LR_GRID = np.logspace(-6, np.log10(0.5), 50)
MUON_LR_GRID = np.logspace(-5, np.log10(0.2), 50)

print("=== Learning Rate Grids ===")
print(f"  SGD  grid: {len(SGD_LR_GRID)} points, range [{SGD_LR_GRID[0]:.2e}, {SGD_LR_GRID[-1]:.2e}]")
print(f"    Spacing factor (consecutive ratio): ~{SGD_LR_GRID[1]/SGD_LR_GRID[0]:.3f}x")
print(f"    First 5 LRs:  {[f'{lr:.2e}' for lr in SGD_LR_GRID[:5]]}")
print(f"    Last 5 LRs:   {[f'{lr:.2e}' for lr in SGD_LR_GRID[-5:]]}")
print(f"")
print(f"  Muon grid: {len(MUON_LR_GRID)} points, range [{MUON_LR_GRID[0]:.2e}, {MUON_LR_GRID[-1]:.2e}]")
print(f"    Spacing factor (consecutive ratio): ~{MUON_LR_GRID[1]/MUON_LR_GRID[0]:.3f}x")
print(f"    First 5 LRs:  {[f'{lr:.2e}' for lr in MUON_LR_GRID[:5]]}")
print(f"    Last 5 LRs:   {[f'{lr:.2e}' for lr in MUON_LR_GRID[-5:]]}")
print(f"")
print("  NOTE: With 50 log-spaced points over ~5 decades (SGD) and ~4 decades (Muon),")
print("  adjacent LRs differ by only ~25-30%. This is fine enough to confidently")
print("  identify the basin of optimal LR, not just a lucky point.")

## Section 3: Core Algorithm Components\n\n### Newton-Schulz Orthogonalization\n\nThe heart of Muon is the Newton-Schulz iteration for computing the polar factor of a matrix.\nGiven gradient G, we want U from the polar decomposition G = U * S (where U is orthogonal\nand S is symmetric positive semi-definite).\n\nThe iteration `X_{k+1} = 1.5 * X_k - 0.5 * X_k @ (X_k^T @ X_k)` converges to the nearest\northogonal matrix when initialized with X_0 = G / ||G||_F.\n\n**Key property:** The resulting update direction has all singular values equal to 1. This\nmeans Muon treats all spectral directions equally -- it \"democratizes\" the gradient across\nsingular value components. SGD, by contrast, is biased toward the top singular directions\nof the gradient, which may or may not align with the most useful descent directions.\n\n### Network Architecture\n\nWe use a deep linear network: y = W_L @ W_{L-1} @ ... @ W_1 @ x, where each W_i is DxD.\nWeights are initialized near identity (I + 0.1 * noise) to start in a well-conditioned regime.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

In [ ]:
def init_weights(depth, seed):
    rng = np.random.RandomState(seed)
    return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(depth)]

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    W_target = rng.randn(DIM, DIM) * 0.5
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = W_target @ X
    return X, Y

### Data Generation and Training Loop\n\nThe target function is a random linear map W_target @ x. The training loop uses\nmomentum SGD (or momentum Muon), with the only difference being whether the\ngradient is passed through Newton-Schulz orthogonalization before accumulation.\n\nThe training function returns `inf` on divergence (loss > 1e10 or NaN), which\nallows the LR sweep to gracefully handle unstable learning rates.

In [ ]:
def train(weights_init, X, Y, lr, optimizer):
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(len(weights)):
            if optimizer == 'muon':
                mom[i] = MOMENTUM * mom[i] + newton_schulz(grads[i])
            else:
                mom[i] = MOMENTUM * mom[i] + grads[i]
            weights[i] = weights[i] - lr * mom[i]
    return compute_loss(weights, X, Y)

### Statistical Tools\n\nWe use Spearman rank correlation to test for monotone trends. If the advantage\ngrows (or shrinks) systematically with depth, Spearman's rho will be close to +1 (or -1).\nWe require |rho| < 0.8 to claim \"no significant trend\" -- a conservative threshold\ngiven only 4 data points (depths).

In [ ]:
def spearman_rank(x, y):
    n = len(x)
    if n < 3:
        return float('nan')
    rx = np.argsort(np.argsort(x)).astype(float) + 1
    ry = np.argsort(np.argsort(y)).astype(float) + 1
    return np.corrcoef(rx, ry)[0, 1]

## Section 4: Main Experiment -- Exhaustive LR Sweep at Each Depth\n\nThe experiment proceeds in two phases per (depth, optimizer) pair:\n\n1. **Selection phase:** For each of 50 LR candidates, run 3 seeds and take the median loss.\n   Select the LR with lowest median. Using median (not mean) provides robustness to outlier\n   seeds that diverge.\n\n2. **Evaluation phase:** At the selected best LR, run all 5 seeds and report the mean loss.\n   This two-phase design avoids overfitting the LR selection to a single lucky seed.\n\nThe advantage ratio is then: `advantage = best_SGD_loss / best_Muon_loss`.\n\nIf the ~30x factor is a genuine property of Newton-Schulz orthogonalization (not an artifact\nof coarse LR tuning), then this ratio should be stable across all 4 depths even with\nexhaustive 50-point LR sweeps.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H18b: IS THE ~30x CONSTANT FACTOR LR-INDEPENDENT?")
    print("=" * 100)
    print(f"Depths: {DEPTHS}")
    print(f"SGD: {len(SGD_LR_GRID)} LR candidates")
    print(f"Muon: {len(MUON_LR_GRID)} LR candidates")
    print(f"\nSeeds used: {seeds}")
    print(f"  Selection phase uses first 3 seeds: {seeds[:3]}")
    print(f"  Evaluation phase uses all {NUM_SEEDS} seeds: {seeds}")
    print()

    advantages = {}
    # Store detailed results for post-hoc analysis
    all_best_lrs = {}
    all_best_losses = {}

    for depth in DEPTHS:
        print(f"\n{'='*80}")
        print(f"  DEPTH L={depth}")
        print(f"{'='*80}")
        print(f"  Network: x -> W1 @ ... @ W{depth} @ x  (product of {depth} matrices, each {DIM}x{DIM})")
        print(f"  Total parameters: {depth * DIM * DIM}")

        best_losses = {}
        for opt, lr_grid in [('sgd', SGD_LR_GRID), ('muon', MUON_LR_GRID)]:
            print(f"\n    --- {opt.upper()} LR Sweep ({len(lr_grid)} candidates) ---")
            best_lr = lr_grid[0]
            best_loss = float('inf')

            # Track all LR -> loss mappings for diagnostics
            lr_loss_map = []

            for lr in lr_grid:
                losses = []
                for s in seeds[:3]:
                    X, Y = make_data(s)
                    w = init_weights(depth, s + 5000)
                    fl = train(w, X, Y, lr, opt)
                    losses.append(fl)
                finite = [l for l in losses if np.isfinite(l)]
                ml = np.median(finite) if finite else float('inf')
                lr_loss_map.append((lr, ml, len(finite)))
                if ml < best_loss:
                    best_loss = ml
                    best_lr = lr

            # Print LR sweep summary
            valid_lrs = [(lr, loss) for lr, loss, nf in lr_loss_map if np.isfinite(loss)]
            diverged_lrs = [(lr, loss) for lr, loss, nf in lr_loss_map if not np.isfinite(loss)]
            print(f"      LR sweep complete: {len(valid_lrs)}/{len(lr_grid)} LRs produced finite losses")
            if diverged_lrs:
                div_range = [lr for lr, _ in diverged_lrs]
                print(f"      Diverged LR range: [{min(div_range):.2e}, {max(div_range):.2e}]")

            # Show top-5 LRs
            sorted_lrs = sorted(valid_lrs, key=lambda x: x[1])
            print(f"      Top-5 LRs by median loss:")
            for rank, (lr, loss) in enumerate(sorted_lrs[:5], 1):
                marker = " <-- BEST" if lr == best_lr else ""
                print(f"        #{rank}: LR={lr:.6e}  median_loss={loss:.6e}{marker}")

            # Show how flat the basin is around the optimum
            if len(sorted_lrs) >= 3:
                top3_lrs = [lr for lr, _ in sorted_lrs[:3]]
                top3_spread = max(top3_lrs) / min(top3_lrs)
                print(f"      Basin width (top-3 LR spread): {top3_spread:.2f}x")

            # Full eval
            all_losses = []
            for s in seeds:
                X, Y = make_data(s)
                w = init_weights(depth, s + 5000)
                fl = train(w, X, Y, best_lr, opt)
                all_losses.append(fl)
            finite = [l for l in all_losses if np.isfinite(l)]
            mean_loss = np.mean(finite) if finite else float('inf')
            best_losses[opt] = mean_loss
            all_best_lrs[(depth, opt)] = best_lr
            all_best_losses[(depth, opt)] = mean_loss

            print(f"\n      FULL EVALUATION at best LR={best_lr:.6e}:")
            print(f"        Per-seed losses: {[f'{l:.4e}' for l in all_losses]}")
            print(f"        Finite losses:   {len(finite)}/{len(all_losses)}")
            if finite:
                print(f"        Mean loss:       {mean_loss:.6e}")
                print(f"        Std loss:        {np.std(finite):.6e}")
                print(f"        Min/Max:         {min(finite):.6e} / {max(finite):.6e}")
            print(f"    {opt:>5}: best_lr={best_lr:.6f}  loss={mean_loss:.6e}")

        adv = best_losses['sgd'] / max(best_losses['muon'], 1e-30)
        advantages[depth] = adv
        print(f"\n    >>> ADVANTAGE RATIO at depth {depth}: {adv:.2f}x")
        print(f"        SGD best loss:  {best_losses['sgd']:.6e}  (at LR={all_best_lrs[(depth, 'sgd')]:.6e})")
        print(f"        Muon best loss: {best_losses['muon']:.6e}  (at LR={all_best_lrs[(depth, 'muon')]:.6e})")
        print(f"        Interpretation: Muon reaches {adv:.1f}x lower loss than SGD,")
        print(f"        even after exhaustive LR tuning for both optimizers.")

    # =========================================================================
    # ANALYSIS
    # =========================================================================
    print(f"\n\n{'='*100}")
    print("RESULTS")
    print(f"{'='*100}")

    adv_list = [advantages[d] for d in DEPTHS]
    log_advs = np.log(adv_list)
    cv = np.std(log_advs) / (np.abs(np.mean(log_advs)) + 1e-15)
    rho = spearman_rank(np.array(DEPTHS, dtype=float), np.array(adv_list))

    print(f"\n  {'Depth':>6}  {'SGD LR':>12}  {'SGD Loss':>12}  {'Muon LR':>12}  {'Muon Loss':>12}  {'Advantage':>12}")
    print("  " + "-" * 80)
    for d in DEPTHS:
        print(f"  {d:>6}  {all_best_lrs[(d,'sgd')]:>12.2e}  {all_best_losses[(d,'sgd')]:>12.4e}  "
              f"{all_best_lrs[(d,'muon')]:>12.2e}  {all_best_losses[(d,'muon')]:>12.4e}  {advantages[d]:>11.1f}x")

    print(f"\n  === Advantage Statistics ===")
    print(f"  Mean advantage: {np.mean(adv_list):.1f}x")
    print(f"  Std advantage:  {np.std(adv_list):.1f}x")
    print(f"  Min advantage:  {min(adv_list):.1f}x  (at depth {DEPTHS[np.argmin(adv_list)]})")
    print(f"  Max advantage:  {max(adv_list):.1f}x  (at depth {DEPTHS[np.argmax(adv_list)]})")
    print(f"  Max/Min ratio:  {max(adv_list)/max(min(adv_list),1e-30):.2f}x")
    print(f"\n  === Log-Space Analysis ===")
    print(f"  log(advantages):      {[f'{la:.3f}' for la in log_advs]}")
    print(f"  Mean log(advantage):  {np.mean(log_advs):.3f}  (= geometric mean advantage of {np.exp(np.mean(log_advs)):.1f}x)")
    print(f"  Std log(advantage):   {np.std(log_advs):.3f}")
    print(f"  CV of log(advantage): {cv:.3f}")
    print(f"\n  === Trend Analysis ===")
    print(f"  Spearman(depth, advantage): {rho:.3f}")
    if abs(rho) < 0.4:
        print(f"    Interpretation: No meaningful correlation -- advantage is essentially independent of depth.")
    elif abs(rho) < 0.8:
        print(f"    Interpretation: Weak correlation present but not strong enough to indicate a systematic trend.")
    else:
        trend_dir = 'increases' if rho > 0 else 'decreases'
        print(f"    Interpretation: Strong monotone trend detected -- the advantage {trend_dir} with depth.")

    # Tests
    all_in_range = all(5 < a < 100 for a in adv_list)
    cv_low = cv < 0.3
    no_trend = abs(rho) < 0.8

    print(f"\n  {'='*60}")
    print(f"  PASS/FAIL CRITERIA")
    print(f"  {'='*60}")
    pass_t1 = 'PASS' if all_in_range else 'FAIL'
    print(f"  T1: All advantages in [5x, 100x]?  --> {pass_t1}")
    if not all_in_range:
        for d in DEPTHS:
            if not (5 < advantages[d] < 100):
                print(f"       FAIL at depth {d}: advantage = {advantages[d]:.1f}x (outside [5, 100])")
    else:
        adv_strs = [f'{a:.1f}x' for a in adv_list]
        print(f"       All values: {adv_strs} are within [5x, 100x]")

    pass_t2 = 'PASS' if cv_low else 'FAIL'
    print(f"  T2: CV of log(advantage) < 0.3?     --> {pass_t2}  (CV={cv:.3f})")
    if cv_low:
        print(f"       Low CV confirms the advantage is consistent across depths in log-space.")
    else:
        print(f"       High CV suggests the advantage varies substantially -- may not be a true constant.")

    pass_t3 = 'PASS' if no_trend else 'FAIL'
    print(f"  T3: No monotone depth trend?         --> {pass_t3}  (rho={rho:.3f})")
    if no_trend:
        print(f"       No significant Spearman correlation -- advantage does not systematically grow/shrink with depth.")
    else:
        print(f"       Significant correlation detected -- the advantage has a depth-dependent component.")

    overall = all_in_range and cv_low and no_trend
    verdict = 'ALL TESTS PASS' if overall else 'SOME TESTS FAILED'
    print(f"\n  OVERALL: {verdict}")

    print(f"\n{'='*100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*100}")

In [ ]:
if __name__ == '__main__':
    main()

## Section 5: Conclusions and Interpretation

### What to look for in the results above

**If ALL TESTS PASS:**
The ~30x Muon advantage is confirmed to be LR-independent. This means:
- The advantage is NOT an artifact of coarse LR tuning
- Even with 50-point exhaustive sweeps, SGD cannot close the gap by finding a better LR
- The Newton-Schulz orthogonalization provides a genuine, fixed-factor improvement in update direction quality
- The constant factor is stable across depths 2-16, ruling out both depth-dependent scaling and depth-independent flukes

**If T1 FAILS (advantage outside [5x, 100x]):**
The advantage may not be a constant factor at all. If some depths show <5x, the Muon benefit
may be marginal and LR-dependent. If some show >100x, there may be a depth-dependent amplification.

**If T2 FAILS (high CV of log-advantage):**
The advantage exists but varies too much across depths to be called a "constant." This would
suggest the benefit depends on network structure, not just the optimization direction.

**If T3 FAILS (significant Spearman trend):**
The advantage grows or shrinks systematically with depth. This would revive the hypothesis of
compounding effects -- possibly gauge-fixing that accumulates layer-by-layer.

### Connection to the Broader Research Program

This experiment is part of a systematic effort to understand WHY Muon works. The findings here
constrain the space of viable theories:

- **Constant factor, LR-independent**: Consistent with "better direction" theory -- the NS
  orthogonalization finds a uniformly better descent direction, worth ~30x in convergence rate
- **Constant factor, LR-dependent**: Would suggest the benefit is really about implicit LR scaling,
  not direction quality (this is what H18b aims to rule out)
- **Depth-dependent factor**: Would point toward gauge-fixing or spectral conditioning effects
  that compound through the network (already investigated and partially ruled out by H18a)